In [ ]:
# --- Environment Representation ---
hospital = {
    "WardA": {"status": "Critical", "adjacent": ["WardB", "WardC"]},
    "WardB": {"status": "Normal", "adjacent": ["WardA", "WardD"]},
    "WardC": {"status": "Critical", "adjacent": ["WardA", "WardD"]},
    "WardD": {"status": "Empty", "adjacent": ["WardB", "WardC"]}
}

# --- Task 1: Create UtilityBasedAgent Class ---
class UtilityBasedAgent:
    def _init_(self, environment, start_ward):
        self.env = environment
        self.current_ward = start_ward
        self.moves = 0
        self.critical_treated = 0

    # Utility function: (10 * critical patients treated) - moves
    def utility(self):
        return (10 * self.critical_treated) - self.moves

    # --- Task 2: Implement BFS Function ---
    def bfs_nearest_critical(self):
        """Finds the shortest path to the nearest 'Critical' ward using a standard list."""
        # Queue as a standard list storing tuples of (current_ward, path_taken)
        queue = [(self.current_ward, [])]
        visited = set([self.current_ward])

        while queue:
            # Using pop(0) acts as a Dequeue operation (FIFO)
            current, path = queue.pop(0)

            # Goal Test: Check if the current ward has a Critical patient
            if self.env[current]["status"] == "Critical":
                return path

            # Expand neighbors
            for neighbor in self.env[current]["adjacent"]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append((neighbor, path + [neighbor]))
                    
        return None # Return None if no critical patients are left

    # --- Task 3 & 4: Agent Actions and Simulation ---
    def run_simulation(self):
        print("=== Drone Simulation Started ===")
        print(f"Starting at: {self.current_ward} | Initial Utility: {self.utility()}\n")

        while True:
            # Action 1: Deliver medicine if the current ward is critical
            if self.env[self.current_ward]["status"] == "Critical":
                print(f"Current Ward: {self.current_ward} | Action: Deliver Medicine")
                
                # Update environment and agent state
                self.env[self.current_ward]["status"] = "Treated"
                self.critical_treated += 1
                
                print(f"Utility: {self.utility()}\n")
                continue # Re-evaluate the current room before moving

            # Action 2: If current is not critical, find the nearest critical ward using BFS
            path_to_next = self.bfs_nearest_critical()

            # If BFS returns None, it means all critical patients are treated
            if not path_to_next:
                print("All critical patients treated. Simulation ending.")
                break

            # Action 3: Move along the BFS path to the next critical ward
            for next_ward in path_to_next:
                self.moves += 1
                self.current_ward = next_ward
                print(f"Current Ward: {self.current_ward} | Action: Move")
                print(f"Utility: {self.utility()}")
                
                # Stop moving if we've reached a critical ward to treat them
                if self.env[self.current_ward]["status"] == "Critical":
                    print() # formatting newline
                    break 

        # Final output required by Task 4
        print("================================")
        print(f"Final Utility: {self.utility()}")
        print(f"Total Moves Taken: {self.moves}")
        print(f"Total Critical Patients Treated: {self.critical_treated}")


# Run the simulation starting from WardB
if _name_ == "_main_":
    drone = UtilityBasedAgent(hospital, start_ward="WardB")
    drone.run_simulation()

In [ ]:
class RescueDroneAStarNoHeap:
    def _init_(self):
        # Grid representation
        # 1 = Safe/Start/Goal, X = Blocked, 3 = Water, 5 = Fire
        self.grid = [
            ['S', '.', '.', 'W', '.'],
            ['.', '#', '.', '#', '.'],
            ['.', '#', '.', '.', '.'],
            ['.', '.', 'F', '#', 'G'],
            ['.', '.', '.', '.', '.']
        ]
        self.rows = len(self.grid)
        self.cols = len(self.grid[0])
        self.start = (0, 0)
        self.goal = (3, 4)

    def get_cost(self, r, c):
        terrain = self.grid[r][c]
        if terrain in ['.', 'S', 'G']: return 1
        if terrain == 'W': return 3
        if terrain == 'F': return 5
        return float('inf') # Blocked area

    def manhattan_distance(self, r, c):
        # h(n) = |current_row - goal_row| + |current_col - goal_col|
        return abs(r - self.goal[0]) + abs(c - self.goal[1])

    def get_neighbors(self, r, c):
        neighbors = []
        # Directions: Up, Down, Left, Right
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            # Check bounds and avoid walls '#'
            if 0 <= nr < self.rows and 0 <= nc < self.cols and self.grid[nr][nc] != '#':
                neighbors.append((nr, nc))
        return neighbors

    def a_star_search(self):
        # Initial heuristic
        start_h = self.manhattan_distance(self.start[0], self.start[1])
        
        # Queue stores tuples: (f_cost, g_cost, current_node, path_taken)
        queue = [(start_h, 0, self.start, [self.start])]
        
        # Track minimum g_cost to reach a node to avoid suboptimal loops
        cost_so_far = {self.start: 0}
        expanded_nodes = []

        while queue:
            # Sort the queue so the lowest f_cost is at index 0
            queue.sort(key=lambda x: x[0])
            
            # Pop the best node (acts like a priority queue)
            f, g, current, path = queue.pop(0)
            
            # Record expansion sequence (only add if we haven't expanded it yet)
            if current not in expanded_nodes:
                expanded_nodes.append(current)

            # Goal Test
            if current == self.goal:
                return path, g, expanded_nodes

            # Expand neighbors
            for neighbor in self.get_neighbors(current[0], current[1]):
                step_cost = self.get_cost(neighbor[0], neighbor[1])
                new_g = g + step_cost
                
                # If we found a cheaper path to this neighbor
                if neighbor not in cost_so_far or new_g < cost_so_far[neighbor]:
                    cost_so_far[neighbor] = new_g
                    
                    # Calculate new heuristic and f_cost
                    h = self.manhattan_distance(neighbor[0], neighbor[1])
                    new_f = new_g + h
                    
                    # Add to queue
                    queue.append((new_f, new_g, neighbor, path + [neighbor]))

        return None, float('inf'), expanded_nodes

# --- Run the Simulation ---
if _name_ == "_main_":
    agent = RescueDroneAStarNoHeap()
    optimal_path, total_cost, expanded = agent.a_star_search()

    print("--- A* Search Results ---")
    print(f"1. Optimal Path: {' -> '.join([str(p) for p in optimal_path])}")
    print(f"2. Total Cost: {total_cost}")
    print(f"3. Expanded Nodes Sequence (First 15): {expanded[:15]}")

In [ ]:
import math
import random

# Input Data from Exam Paper
cities = {
    0: (2, 3),
    1: (5, 7),
    2: (1, 9),
    3: (8, 2),
    4: (4, 5)
}

# --- Task 1: Calculate total distance [02 Marks] ---
def calculate_total_distance(route):
    """Calculates the total Euclidean distance of the route, returning to the start."""
    total_dist = 0
    num_cities = len(route)
    for i in range(num_cities):
        # Current city and next city (using modulo to wrap around to the start)
        c1 = cities[route[i]]
        c2 = cities[route[(i + 1) % num_cities]]
        # Euclidean distance formula
        total_dist += math.sqrt((c2[0] - c1[0])*2 + (c2[1] - c1[1])*2)
    return total_dist

# --- Task 2: Tournament Selection [03 Marks] ---
def tournament_selection(population, tournament_size=3):
    """Selects the best individual from a randomly chosen subset."""
    best = None
    for _ in range(tournament_size):
        contender = random.choice(population)
        if best is None or calculate_total_distance(contender) < calculate_total_distance(best):
            best = contender
    return best

# --- Task 3: Order Crossover (OX) [04 Marks] ---
def order_crossover(parent1, parent2):
    """Performs Order Crossover to preserve valid permutations (no duplicate cities)."""
    size = len(parent1)
    child = [-1] * size
    
    # 1. Select a random swath (start and end indices)
    start, end = sorted(random.sample(range(size), 2))
    
    # 2. Copy the swath from parent 1 to the child
    child[start:end+1] = parent1[start:end+1]
    
    # 3. Fill the remaining spots with cities from parent 2, preserving order
    pointer = 0
    for city in parent2:
        if city not in child:
            # Find the next available empty spot (-1)
            while child[pointer] != -1:
                pointer += 1
            child[pointer] = city
            
    return child

# --- Task 4: Mutation (Swap) [03 Marks] ---
def swap_mutation(route, mutation_rate=0.2):
    """Randomly swaps two cities in the route to maintain genetic diversity."""
    if random.random() < mutation_rate:
        idx1, idx2 = random.sample(range(len(route)), 2)
        # Swap the alleles
        route[idx1], route[idx2] = route[idx2], route[idx1]
    return route

# --- Task 5: Main Execution & Stopping Criteria [06 Marks] ---
def run_genetic_algorithm(num_cities, max_generations=100, target_distance=23.88):
    # Initialize a random population of routes (permutations of city indices)
    population_size = 20
    base_route = list(range(num_cities))
    population = [random.sample(base_route, num_cities) for _ in range(population_size)]
    
    best_route = None
    best_distance = float('inf')

    # Stopping Criteria 1: Predefined number of generations
    for generation in range(max_generations):
        new_population = []
        
        # Create new generation
        for _ in range(population_size):
            # Selection
            p1 = tournament_selection(population)
            p2 = tournament_selection(population)
            
            # Crossover
            child = order_crossover(p1, p2)
            
            # Mutation
            child = swap_mutation(child)
            
            new_population.append(child)
            
        population = new_population
        
        # Evaluate current population to find the best
        for route in population:
            dist = calculate_total_distance(route)
            if dist < best_distance:
                best_distance = dist
                best_route = route
        
        # Stopping Criteria 2: Desired minimal distance found
        if best_distance <= target_distance:
            print(f"Target distance reached early at Generation {generation}!")
            break

    return best_route, best_distance

# --- Execution to match Exam Output ---
if _name_ == "_main_":
    route, dist = run_genetic_algorithm(num_cities=5)
    
    print("\nOutput:")
    print(f"Best Route: {route}")
    print(f"Total Distance: {round(dist, 2)} units")

In [ ]:
class UtilityBasedAgent:
    def __init__(self, graph, burning_zones, start_zone):
        self.graph = graph
        self.burning_zones = burning_zones[:]  # mutable copy
        self.current_zone = start_zone
        self.moves = 0
        self.fires_extinguished = 0
        self.path = []

    def utility(self):
        return 10 * self.fires_extinguished - self.moves

    def find_path_dfs(self, start, targets):
        visited = set()
        parent = {start: None}

        def dfs(node):
            if node in targets:
                return True
            for neighbor in self.graph.get(node, []):
                if neighbor not in visited:
                    visited.add(neighbor)
                    parent[neighbor] = node
                    if dfs(neighbor):
                        return True
            return False

        visited.add(start)
        if dfs(start):
            path = []
            cur = start
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            path.reverse()
            return path
        return None

    def act(self):
        if self.current_zone in self.burning_zones:
            self.fires_extinguished += 1
            self.burning_zones.remove(self.current_zone)
            return "Extinguish"
        else:
            if not self.path or self.path[0] != self.current_zone:
                self.path = self.find_path_dfs(self.current_zone, self.burning_zones)
            if self.path and len(self.path) > 1:
                next_zone = self.path[1]
                self.current_zone = next_zone
                self.moves += 1
                return f"Move to {next_zone}"
            return "No action"


forest_graph = {
    'ZoneA': ['ZoneB', 'ZoneC'],
    'ZoneB': ['ZoneA', 'ZoneD'],
    'ZoneC': ['ZoneA', 'ZoneE'],
    'ZoneD': ['ZoneB'],
    'ZoneE': ['ZoneC']
}
burning_zones = ['ZoneD', 'ZoneE']

agent = UtilityBasedAgent(forest_graph, burning_zones, 'ZoneA')

while agent.burning_zones:
    action = agent.act()
    print(f"Current Zone: {agent.current_zone}, Action: {action}, Utility: {agent.utility()}")

print(f"Final Utility: {agent.utility()}")

In [4]:
import heapq

# Grid representation
grid = [
    ['S', '.', 'F', '.', '.'],
    ['W', '#', 'W', '#', '.'],
    ['.', '.', '.', 'F', '.'],
    ['#', '.', '.', '.', 'G'],
    ['.', 'W', '.', '.', '.']
]

rows = len(grid)
cols = len(grid[0])

# Cost mapping
terrain_cost = {
    'S': 0,    # start cell, no cost to enter
    '.': 1,
    'G': 1,    # goal cell costs 1 to enter
    'W': 3,
    'F': 5,
    '#': None  # blocked
}

# Directions: up, down, left, right
directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

# Find start and goal positions
start = None
goal = None
for r in range(rows):
    for c in range(cols):
        if grid[r][c] == 'S':
            start = (r, c)
        elif grid[r][c] == 'G':
            goal = (r, c)

def heuristic(r, c):
    """Manhattan distance to goal."""
    return abs(r - goal[0]) + abs(c - goal[1])

def a_star():
    # Priority queue: (f, g, r, c)
    pq = []
    heapq.heappush(pq, (0, 0, start[0], start[1]))
    
    # Tracking best known g values
    g_score = {start: 0}
    parent = {start: None}
    expanded = []  # order of expansion
    
    while pq:
        f, g, r, c = heapq.heappop(pq)
        
        # Skip outdated entries
        if (r, c) in g_score and g > g_score[(r, c)]:
            continue
        
        expanded.append((r, c))
        
        if (r, c) == goal:
            # Reconstruct path
            path = []
            cur = goal
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            path.reverse()
            return expanded, path, g
        
        # Explore neighbors
        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if not (0 <= nr < rows and 0 <= nc < cols):
                continue
            cell = grid[nr][nc]
            cost = terrain_cost.get(cell)
            if cost is None:  # blocked
                continue
            new_g = g + cost
            if (nr, nc) not in g_score or new_g < g_score[(nr, nc)]:
                g_score[(nr, nc)] = new_g
                parent[(nr, nc)] = (r, c)
                h = heuristic(nr, nc)
                heapq.heappush(pq, (new_g + h, new_g, nr, nc))
    
    return expanded, None, None

expanded, path, total_cost = a_star()

print("Nodes expanded (in order):")
for node in expanded:
    print(node)
print("\nOptimal path:")
for node in path:
    print(node)
print(f"\nTotal cost: {total_cost}")

Nodes expanded (in order):
(0, 0)
(0, 1)
(1, 0)
(2, 0)
(2, 1)
(2, 2)
(3, 1)
(3, 2)
(3, 3)
(3, 4)

Optimal path:
(0, 0)
(1, 0)
(2, 0)
(2, 1)
(2, 2)
(3, 2)
(3, 3)
(3, 4)

Total cost: 9


In [5]:
import random
import math

# TSP Genetic Algorithm
def distance(city1, city2):
    return math.sqrt((city1[0] - city2[0])**2 + (city1[1] - city2[1])**2)

def total_distance(route, cities):
    dist = 0
    for i in range(len(route)):
        dist += distance(cities[route[i]], cities[route[(i+1) % len(route)]])
    return dist

def create_individual(n):
    ind = list(range(n))
    random.shuffle(ind)
    return ind

def roulette_wheel_selection(population, fitness_values):
    total_fitness = sum(fitness_values)
    pick = random.uniform(0, total_fitness)
    cumulative = 0
    for i, fit in enumerate(fitness_values):
        cumulative += fit
        if cumulative >= pick:
            return population[i]
    return population[-1]

def two_point_crossover(parent1, parent2):
    n = len(parent1)
    p1, p2 = sorted(random.sample(range(1, n), 2))
    child = [-1] * n
    child[p1:p2+1] = parent1[p1:p2+1]
    fill_pos = 0
    for gene in parent2:
        if gene not in child:
            while child[fill_pos] != -1:
                fill_pos += 1
            child[fill_pos] = gene
    return child

def inversion_mutation(individual):
    n = len(individual)
    i, j = sorted(random.sample(range(n), 2))
    individual[i:j+1] = reversed(individual[i:j+1])
    return individual

def genetic_algorithm(cities, pop_size=100, generations=500, mutation_rate=0.2, target_distance=None):
    n = len(cities)
    population = [create_individual(n) for _ in range(pop_size)]
    best_route = None
    best_dist = float('inf')
    
    for gen in range(generations):
        # Evaluate fitness (inverse distance, because we want to maximize fitness for selection)
        distances = [total_distance(route, cities) for route in population]
        # For roulette wheel, we use fitness as inverse distance (higher fitness = shorter distance)
        # Avoid division by zero; add small epsilon
        fitness = [1.0 / (d + 1e-6) for d in distances]
        
        # Update best
        min_idx = distances.index(min(distances))
        if distances[min_idx] < best_dist:
            best_dist = distances[min_idx]
            best_route = population[min_idx][:]
        
        # Termination: if target distance reached
        if target_distance and best_dist <= target_distance:
            print(f"Target distance reached at generation {gen+1}")
            break
        
        # Selection
        new_population = []
        while len(new_population) < pop_size:
            p1 = roulette_wheel_selection(population, fitness)
            p2 = roulette_wheel_selection(population, fitness)
            child = two_point_crossover(p1, p2)
            if random.random() < mutation_rate:
                child = inversion_mutation(child)
            new_population.append(child)
        population = new_population
    
    return best_route, best_dist

# Input
cities = [(2,3), (5,7), (1,9), (8,2), (4,5)]
n = len(cities)

route, dist = genetic_algorithm(cities, pop_size=200, generations=1000, mutation_rate=0.3)
print("Best Route:", route)
print("Total Distance:", round(dist, 2))

Best Route: [3, 0, 2, 1, 4]
Total Distance: 23.87


In [1]:
class SimpleReflexLightAgent:
    def __init__(self):
        pass
    
    def act(self, occupancy_status):
        actions = {}
        for room, occupied in occupancy_status.items():
            if occupied:
                actions[room] = "Turn ON"
            else:
                actions[room] = "Turn OFF"
        return actions

occupancy = {
    "Room-1" : False,
    "Room-2" : True,
    "Room-3" : False,
    "Room-4" : True,
    "Room-5" : False
}

agent = SimpleReflexLightAgent()

actions = agent.act(occupancy)

for room in occupancy:
    print(f"{room} : Occupied = {occupancy[room]}, Action = {actions[room]}")

Room-1 : Occupied = False, Action = Turn OFF
Room-2 : Occupied = True, Action = Turn ON
Room-3 : Occupied = False, Action = Turn OFF
Room-4 : Occupied = True, Action = Turn ON
Room-5 : Occupied = False, Action = Turn OFF


In [2]:
class ModelBasedWarehouseAgent:
    def __init__(self):
        self.model = {} 
        self.location = "A1"
        self.pick_list = ["item1", "item3"] 
        self.collected = []
    
    def update_model(self, percept):
        shelf, has_item, item_type = percept
        self.model[shelf] = (has_item, item_type)
    
    def act(self, percept):
        shelf, has_item, item_type = percept
        
        self.update_model((shelf, has_item, item_type))
        
        if has_item and item_type in self.pick_list and item_type not in self.collected:
            self.collected.append(item_type)
            return "Pick"
        
        if len(self.collected) == len(self.pick_list):
            return "Deliver"
        
        if self.location == "A1":
            self.location = "A2"
        elif self.location == "A2":
            self.location = "A3"
        else:
            self.location = "A1"
        
        return "Move"

environment = {
    "A1": (True, "item1"),
    "A2": (False, None),
    "A3": (True, "item3")
}

agent = ModelBasedWarehouseAgent()

for _ in range(6):
    shelf = agent.location
    has_item, item_type = environment[shelf]
    action = agent.act((shelf, has_item, item_type))
    
    print(f"Shelf: {shelf}, Item: {item_type}, Action: {action}")
    
    if action == "Pick":
        environment[shelf] = (False, None)

Shelf: A1, Item: item1, Action: Pick
Shelf: A1, Item: None, Action: Move
Shelf: A2, Item: None, Action: Move
Shelf: A3, Item: item3, Action: Pick
Shelf: A3, Item: None, Action: Deliver
Shelf: A3, Item: None, Action: Deliver


In [3]:
class ModelBasedAgent:
    def __init__(self):
        
        self.model = {
            "RoomA": "Unknown",
            "RoomB": "Unknown"
        }
        self.current_room = "RoomA"

    def update_model(self, percept):
        
        self.model[self.current_room] = percept

    def act(self, percept):
        self.update_model(percept)

        if percept == "Dirty":
            return "Suck"
        else:
            if self.current_room == "RoomA":
                self.current_room = "RoomB"
            else:
                self.current_room = "RoomA"
            return "Move"


environment = {
    "RoomA": "Dirty",
    "RoomB": "Clean"
}


agent = ModelBasedAgent()


for _ in range(4):
    room = agent.current_room
    percept = environment[room]
    action = agent.act(percept)

    print(f"Room: {room}, Percept: {percept}, Action: {action}")
   
    if action == "Suck":
        environment[room] = "Clean"

Room: RoomA, Percept: Dirty, Action: Suck
Room: RoomA, Percept: Clean, Action: Move
Room: RoomB, Percept: Clean, Action: Move
Room: RoomA, Percept: Clean, Action: Move


In [4]:
class GoalBasedAgent:
    def __init__(self, environment, goal):
        self.environment = environment  
        self.goal = goal                
        self.current_room = "RoomA"

    def is_goal_achieved(self):
        
        return all(self.environment[room] == self.goal[room] for room in self.environment)

    def act(self):
        
        percept = self.environment[self.current_room]

        if percept == "Dirty":
            
            self.environment[self.current_room] = "Clean"
            return "Suck"
        else:
            
            for room in self.environment:
                if self.environment[room] == "Dirty":
                    self.current_room = room
                    return f"Move to {room}"
            
            return "No Action, Goal Achieved"

environment = {"RoomA": "Dirty", "RoomB": "Dirty"}
goal = {"RoomA": "Clean", "RoomB": "Clean"}

agent = GoalBasedAgent(environment, goal)

while not agent.is_goal_achieved():
    action = agent.act()
    print(f"Current Room: {agent.current_room}, Action: {action}")

print("Goal Achieved! All rooms are clean")

Current Room: RoomA, Action: Suck
Current Room: RoomB, Action: Move to RoomB
Current Room: RoomB, Action: Suck
Goal Achieved! All rooms are clean


In [5]:
class UtilityBasedAgent:
    def __init__(self, environment):
        self.environment = environment
        self.current_room = "RoomA"
        self.moves = 0 

    def utility(self):
        clean_rooms = sum(1 for room in self.environment if self.environment[room] == "Clean")
        return clean_rooms - self.moves

    def act(self):
        percept = self.environment[self.current_room]

        if percept == "Dirty":
            self.environment[self.current_room] = "Clean"
            return "Suck"
        else:
            
            dirty_rooms = [room for room in self.environment if self.environment[room] == "Dirty"]
            if dirty_rooms:
                
                self.current_room = dirty_rooms[0]
                self.moves += 1
                return f"Move to {self.current_room}"
            else:
                return "No Action, All Clean"

environment = {"RoomA": "Dirty", "RoomB": "Dirty", "RoomC": "Clean"}
agent = UtilityBasedAgent(environment)

while any(environment[room] == "Dirty" for room in environment):
    action = agent.act()
    print(f"Current Room: {agent.current_room}, Action: {action}, Utility: {agent.utility()}")

print("All rooms clean! Final Utility:", agent.utility())

Current Room: RoomA, Action: Suck, Utility: 2
Current Room: RoomB, Action: Move to RoomB, Utility: 1
Current Room: RoomB, Action: Suck, Utility: 2
All rooms clean! Final Utility: 2


In [6]:
import random

class LearningBasedAgent:
    def __init__(self, actions):
        self.Q = {}
        self.actions = actions
        self.alpha = 0.1 
        self.gamma = 0.9 
        self.epsilon = 0.1 
    
    def get_Q_value(self, state, action):
        return self.Q.get((state, action), 0.0)
    
    def select_action(self, state):
        if random.uniform(0, 1) < self.epsilon:
            return random.choice(self.actions)
        else:
            return max(self.actions, key=lambda a: self.get_Q_value(state, a))
    
    def learn(self, state, action, reward, next_state):
        old_Q = self.get_Q_value(state, action)
        best_future_Q = max([self.get_Q_value(next_state, a) for a in self.actions])
        
        self.Q[(state, action)] = old_Q + self.alpha * (reward + self.gamma * best_future_Q - old_Q)
    
    def act(self, state):
        action = self.select_action(state)
        return action


class Environment:
    def __init__(self, state='Dirty'):
        self.state = state
    
    def get_percept(self):
        return self.state
    
    def clean_room(self):
        self.state = 'Clean'
        return 10
    
    def no_action_reward(self):
        return 0


def run_agent(agent, environment, steps):
    for step in range(steps):
        percept = environment.get_percept()
        action = agent.act(percept)
        
        if percept == 'Dirty':
            reward = environment.clean_room()
            print(f"Step {step + 1}: Percept - {percept}, Action - {action}, Reward - {reward}")
        else:
            reward = environment.no_action_reward()
            print(f"Step {step + 1}: Percept - {percept}, Action - {action}, Reward - {reward}")
        
        next_percept = environment.get_percept()
        agent.learn(percept, action, reward, next_percept)

agent = LearningBasedAgent(['Clean the room', 'No action needed'])
environment = Environment()

run_agent(agent, environment, 5)

Step 1: Percept - Dirty, Action - Clean the room, Reward - 10
Step 2: Percept - Clean, Action - Clean the room, Reward - 0
Step 3: Percept - Clean, Action - Clean the room, Reward - 0
Step 4: Percept - Clean, Action - No action needed, Reward - 0
Step 5: Percept - Clean, Action - Clean the room, Reward - 0


In [7]:
graph = {
    "Control Room": ["Storage A", "Storage B"],
    "Storage A": ["Lab 1", "Lab 2"],
    "Storage B": ["Server Room"],
    "Lab 1": ["Debris 1"],
    "Lab 2": ["Debris 2"],
    "Debris 1": ["Maintenance Corridor"],
    "Debris 2": ["Maintenance Corridor"],
    "Server Room": ["Charging Bay"],
    "Charging Bay": ["Maintenance Corridor"],
    "Maintenance Corridor": ["Survivor Room"],
    "Survivor Room": []
}

def bfs(graph, start, goal):
    visited = set()
    queue = [start]
    visited.add(start)
    parent = {start: None}
    expansion_order = []

    while queue:
        node = queue.pop(0)
        expansion_order.append(node)

        if node == goal:
            path = []
            curr = node
            while curr is not None:
                path.append(curr)
                curr = parent[curr]
            path.reverse()
            return expansion_order, path

        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = node
                queue.append(neighbor)

    return expansion_order, None

start_node = "Control Room"
goal_node = "Survivor Room"
order, path = bfs(graph, start_node, goal_node)

print("BFS Order of node expansion:", order)
print("BFS Path to goal:", path)

BFS Order of node expansion: ['Control Room', 'Storage A', 'Storage B', 'Lab 1', 'Lab 2', 'Server Room', 'Debris 1', 'Debris 2', 'Charging Bay', 'Maintenance Corridor', 'Survivor Room']
BFS Path to goal: ['Control Room', 'Storage A', 'Lab 1', 'Debris 1', 'Maintenance Corridor', 'Survivor Room']


In [8]:
graph = {
    "Control Room": ["Storage A", "Storage B"],
    "Storage A": ["Lab 1", "Lab 2"],
    "Storage B": ["Server Room"],
    "Lab 1": ["Debris 1"],
    "Lab 2": ["Debris 2"],
    "Debris 1": ["Maintenance Corridor"],
    "Debris 2": ["Maintenance Corridor"],
    "Server Room": ["Charging Bay"],
    "Charging Bay": ["Maintenance Corridor"],
    "Maintenance Corridor": ["Survivor Room"],
    "Survivor Room": []
}

def dfs_recursive(node, goal, visited, parent, expansion):
    visited.add(node)
    expansion.append(node)

    if node == goal:
        return True

    for neighbor in graph.get(node, []):
        if neighbor not in visited:
            parent[neighbor] = node
            if dfs_recursive(neighbor, goal, visited, parent, expansion):
                return True
    return False

start_node = "Control Room"
goal_node = "Survivor Room"
visited_set = set()
parent_dict = {start_node: None}
expansion_order = []

found = dfs_recursive(start_node, goal_node, visited_set, parent_dict, expansion_order)

if found:
    path = []
    curr = goal_node
    while curr is not None:
        path.append(curr)
        curr = parent_dict[curr]
    path.reverse()
else:
    path = None

print("DFS Recursive Order of node expansion:", expansion_order)
print("DFS Recursive Path to goal:", path)

DFS Recursive Order of node expansion: ['Control Room', 'Storage A', 'Lab 1', 'Debris 1', 'Maintenance Corridor', 'Survivor Room']
DFS Recursive Path to goal: ['Control Room', 'Storage A', 'Lab 1', 'Debris 1', 'Maintenance Corridor', 'Survivor Room']


In [9]:
graph = {
    "Control Room": ["Storage A", "Storage B"],
    "Storage A": ["Lab 1", "Lab 2"],
    "Storage B": ["Server Room"],
    "Lab 1": ["Debris 1"],
    "Lab 2": ["Debris 2"],
    "Debris 1": ["Maintenance Corridor"],
    "Debris 2": ["Maintenance Corridor"],
    "Server Room": ["Charging Bay"],
    "Charging Bay": ["Maintenance Corridor"],
    "Maintenance Corridor": ["Survivor Room"],
    "Survivor Room": []
}

def dfs_iterative(graph, start, goal):
    visited = set()
    stack = [start]
    visited.add(start)
    parent = {start: None}
    expansion_order = []

    while stack:
        node = stack.pop()
        expansion_order.append(node)

        if node == goal:
            path = []
            curr = node
            while curr is not None:
                path.append(curr)
                curr = parent[curr]
            path.reverse()
            return expansion_order, path

        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                parent[neighbor] = node
                stack.append(neighbor)

    return expansion_order, None

start_node = "Control Room"
goal_node = "Survivor Room"
order, path = dfs_iterative(graph, start_node, goal_node)

print("DFS Iterative Order of node expansion:", order)
print("DFS Iterative Path to goal:", path)

DFS Iterative Order of node expansion: ['Control Room', 'Storage B', 'Server Room', 'Charging Bay', 'Maintenance Corridor', 'Survivor Room']
DFS Iterative Path to goal: ['Control Room', 'Storage B', 'Server Room', 'Charging Bay', 'Maintenance Corridor', 'Survivor Room']


In [10]:
graph = {
    "Central Command": [("Shelter A", 4), ("Shelter B", 2)],
    "Shelter A": [("Street-1", 5), ("Street-2", 3)],
    "Shelter B": [("Street-2", 2)],
    "Street-1": [("Hospital Zone", 7)],
    "Street-2": [("Hospital Zone", 4), ("Street-3", 6)],
    "Street-3": [("Hospital Zone", 2)],
    "Hospital Zone": []
}

def ucs(graph, start, goal):
    frontier = [(0, start)]
    best_cost = {start: 0}
    parent = {start: None}
    expansion_order = []

    while frontier:
        min_index = 0
        for i in range(1, len(frontier)):
            if frontier[i][0] < frontier[min_index][0]:
                min_index = i
        cost, node = frontier.pop(min_index)

        if cost > best_cost[node]:
            continue

        expansion_order.append(node)

        if node == goal:
            path = []
            curr = node
            while curr is not None:
                path.append(curr)
                curr = parent[curr]
            path.reverse()
            return expansion_order, path, cost

        for neighbor, edge_cost in graph.get(node, []):
            new_cost = cost + edge_cost
            if neighbor not in best_cost or new_cost < best_cost[neighbor]:
                best_cost[neighbor] = new_cost
                parent[neighbor] = node
                frontier.append((new_cost, neighbor))

    return expansion_order, None, None

start = "Central Command"
goal = "Hospital Zone"
order, path, total_cost = ucs(graph, start, goal)

print("UCS Order of node expansion:", order)
print("UCS Path to goal:", path)
print("Total cost:", total_cost)

UCS Order of node expansion: ['Central Command', 'Shelter B', 'Shelter A', 'Street-2', 'Hospital Zone']
UCS Path to goal: ['Central Command', 'Shelter B', 'Street-2', 'Hospital Zone']
Total cost: 8


In [11]:
graph = {
    "Entrance": [("Room A", 2), ("Room B", 3)],
    "Room A": [("Room C", 2), ("Room D", 5)],
    "Room B": [("Room D", 2)],
    "Room C": [("Treasure Room 1", 4)],
    "Room D": [("Treasure Room 2", 3), ("Trap Room", 1)],
    "Trap Room": [("Treasure Room 3", 2)],
    "Treasure Room 1": [],
    "Treasure Room 2": [],
    "Treasure Room 3": []
}

treasure_values = {
    "Treasure Room 1": 10,
    "Treasure Room 2": 8,
    "Treasure Room 3": 15
}

def ucs_treasure_hunt(graph, start, treasure_values):
    frontier = [(0, start)]
    best_cost = {start: 0}
    parent = {start: None}
    expansion_order = []
    treasure_costs = {}

    while frontier:
        min_index = 0
        for i in range(1, len(frontier)):
            if frontier[i][0] < frontier[min_index][0]:
                min_index = i
        cost, node = frontier.pop(min_index)

        if cost > best_cost[node]:
            continue

        expansion_order.append(node)

        if node in treasure_values:
            treasure_costs[node] = cost

        for neighbor, edge_cost in graph.get(node, []):
            new_cost = cost + edge_cost
            if neighbor not in best_cost or new_cost < best_cost[neighbor]:
                best_cost[neighbor] = new_cost
                parent[neighbor] = node
                frontier.append((new_cost, neighbor))

    best_utility = float('-inf')
    best_treasure = None
    for treasure, cost in treasure_costs.items():
        utility = treasure_values[treasure] - cost
        if utility > best_utility:
            best_utility = utility
            best_treasure = treasure

    path = []
    if best_treasure:
        curr = best_treasure
        while curr is not None:
            path.append(curr)
            curr = parent[curr]
        path.reverse()

    return expansion_order, path, best_utility

def compute_path_cost(graph, path):
    if len(path) < 2:
        return 0
    total = 0
    for i in range(len(path) - 1):
        for neighbor, cost in graph[path[i]]:
            if neighbor == path[i + 1]:
                total += cost
                break
    return total

start = "Entrance"
order, path, utility = ucs_treasure_hunt(graph, start, treasure_values)
cost = compute_path_cost(graph, path)
treasure = path[-1]
value = treasure_values[treasure]

print("Node expansion order:", order)
print("Path to goal:", " -> ".join(path))
print(f"Total utility: {value} - {cost} = {utility}")

Node expansion order: ['Entrance', 'Room A', 'Room B', 'Room C', 'Room D', 'Trap Room', 'Treasure Room 1', 'Treasure Room 2', 'Treasure Room 3']
Path to goal: Entrance -> Room B -> Room D -> Trap Room -> Treasure Room 3
Total utility: 15 - 8 = 7


In [12]:
from collections import deque

grid = [
    ['S', 1, '#', 2, 3],
    [2, '#', 2, '#', 1],
    [3, 2, 1, 2, 2],
    ['#', 2, '#', 1, 3],
    [2, 1, 2, 2, 'G']
]

rows, cols = 5, 5
start = (0, 0)
goal = (4, 4)

# Directions: up, down, left, right
directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def bfs():
    queue = deque([start])
    visited = set([start])
    parent = {start: None}
    expansion_order = []

    while queue:
        r, c = queue.popleft()
        expansion_order.append((r, c))

        if (r, c) == goal:
            # Reconstruct path
            path = []
            cur = goal
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            path.reverse()
            return expansion_order, path

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited:
                if grid[nr][nc] != '#':  # not an obstacle
                    visited.add((nr, nc))
                    parent[(nr, nc)] = (r, c)
                    queue.append((nr, nc))

    return expansion_order, None

order, path = bfs()
print("BFS Expansion order:", order)
print("BFS Path:", path)

BFS Expansion order: [(0, 0), (1, 0), (0, 1), (2, 0), (2, 1), (3, 1), (2, 2), (4, 1), (1, 2), (2, 3), (4, 0), (4, 2), (3, 3), (2, 4), (4, 3), (3, 4), (1, 4), (4, 4)]
BFS Path: [(0, 0), (1, 0), (2, 0), (2, 1), (3, 1), (4, 1), (4, 2), (4, 3), (4, 4)]


In [13]:
grid = [
    ['S', 1, '#', 2, 3],
    [2, '#', 2, '#', 1],
    [3, 2, 1, 2, 2],
    ['#', 2, '#', 1, 3],
    [2, 1, 2, 2, 'G']
]

rows, cols = 5, 5
start = (0, 0)
goal = (4, 4)

directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def dfs():
    stack = [start]
    visited = set([start])
    parent = {start: None}
    expansion_order = []

    while stack:
        r, c = stack.pop()
        expansion_order.append((r, c))

        if (r, c) == goal:
            path = []
            cur = goal
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            path.reverse()
            return expansion_order, path

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited:
                if grid[nr][nc] != '#':
                    visited.add((nr, nc))
                    parent[(nr, nc)] = (r, c)
                    stack.append((nr, nc))

    return expansion_order, None

order, path = dfs()
print("DFS Expansion order:", order)
print("DFS Path:", path)

DFS Expansion order: [(0, 0), (0, 1), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]
DFS Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]


In [14]:
import heapq

grid = [
    ['S', 1, '#', 2, 3],
    [2, '#', 2, '#', 1],
    [3, 2, 1, 2, 2],
    ['#', 2, '#', 1, 3],
    [2, 1, 2, 2, 'G']
]

rows, cols = 5, 5
start = (0, 0)
goal = (4, 4)

directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def cost_of_cell(r, c):
    """Return the cost to enter this cell (start and goal cost 0)."""
    val = grid[r][c]
    if val == 'S' or val == 'G':
        return 0
    return int(val)

def ucs():
    # priority queue: (total_cost, row, col)
    pq = [(0, start[0], start[1])]
    best_cost = {start: 0}
    parent = {start: None}
    expansion_order = []

    while pq:
        cost, r, c = heapq.heappop(pq)

        if (r, c) in best_cost and cost > best_cost[(r, c)]:
            continue

        expansion_order.append((r, c))

        if (r, c) == goal:
            path = []
            cur = goal
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            path.reverse()
            return expansion_order, path, cost

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != '#':
                new_cost = cost + cost_of_cell(nr, nc)
                if (nr, nc) not in best_cost or new_cost < best_cost[(nr, nc)]:
                    best_cost[(nr, nc)] = new_cost
                    parent[(nr, nc)] = (r, c)
                    heapq.heappush(pq, (new_cost, nr, nc))

    return expansion_order, None, None

order, path, total_cost = ucs()
print("UCS Expansion order:", order)
print("UCS Path:", path)
print("Total cost:", total_cost)

UCS Expansion order: [(0, 0), (0, 1), (1, 0), (2, 0), (2, 1), (2, 2), (3, 1), (1, 2), (2, 3), (4, 1), (3, 3), (2, 4), (4, 0), (4, 2), (1, 4), (4, 3), (4, 4)]
UCS Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (3, 3), (4, 3), (4, 4)]
Total cost: 13


In [15]:
import heapq

# Grid: rows A-E (0-4), columns 1-5 (0-4)
grid = [
    ['S', 4, '#', 2, 0],
    [3, '#', 3, '#', 1],
    [2, 2, 2, 2, 1],
    ['#', 3, '#', 1, 2],
    [3, 2, 3, 2, 'G']
]

rows, cols = 5, 5
start = (0, 0)
goal = (4, 4)

# Directions: up, down, left, right
directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def heuristic(r, c):
    """Return heuristic value for a cell (goal = 0, start = 0)."""
    if (r, c) == goal:
        return 0
    val = grid[r][c]
    if val == 'S' or val == '#':
        return 0  # start has no heuristic; obstacles are never visited
    return int(val)

def greedy_best_first():
    # Priority queue: (heuristic, row, col)
    pq = [(heuristic(start[0], start[1]), start[0], start[1])]
    visited = set()
    parent = {start: None}
    expansion_order = []

    while pq:
        h, r, c = heapq.heappop(pq)
        if (r, c) in visited:
            continue
        visited.add((r, c))
        expansion_order.append((r, c))

        if (r, c) == goal:
            # Reconstruct path
            path = []
            cur = goal
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            path.reverse()
            return expansion_order, path

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                if grid[nr][nc] != '#' and (nr, nc) not in visited:
                    if (nr, nc) not in parent:
                        parent[(nr, nc)] = (r, c)
                    heapq.heappush(pq, (heuristic(nr, nc), nr, nc))

    return expansion_order, None

order, path = greedy_best_first()
print("Greedy Best-First Expansion order:", order)
print("Greedy Best-First Path:", path)

Greedy Best-First Expansion order: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (1, 4), (0, 4), (3, 3), (0, 3), (3, 4), (4, 4)]
Greedy Best-First Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4)]


In [16]:
import random
N = 7
POPULATION_SIZE = 100
MAX_GENERATIONS = 1000
MUTATION_RATE = 0.1
TOURNAMENT_SIZE = 3
MAX_FITNESS = N * (N - 1) // 2   # 21
# PROCESS 1: Chromosome Representation (Random Permutation)
def create_chromosome():
    return random.sample(range(N), N)
# PROCESS 2: Fitness Function (Count Non-Attacking Pairs)
def fitness(chromosome):
    non_attacking = 0
    for i in range(N):
        for j in range(i + 1, N):
            if abs(chromosome[i] - chromosome[j]) != abs(i - j):
                non_attacking += 1
    return non_attacking
# PROCESS 3: Selection (Tournament Selection)
def tournament_selection(population):
    selected = []
    for _ in range(2):  # select two parents
        tournament = random.sample(population, TOURNAMENT_SIZE)
        best = max(tournament, key=fitness)
        selected.append(best)
    return selected[0], selected[1]
# PROCESS 4: Crossover (Order Crossover - OX)
def order_crossover(parent1, parent2):
    size = len(parent1)
    start, end = sorted(random.sample(range(size), 2))
    child = [None] * size

    child[start:end+1] = parent1[start:end+1]

    current = 0
    for gene in parent2:
        if gene not in child:
            while child[current] is not None:
                current += 1
            child[current] = gene
    return child
# PROCESS 5: Mutation (Swap Mutation)
def swap_mutation(chromosome):
    idx1, idx2 = random.sample(range(N), 2)
    chromosome[idx1], chromosome[idx2] = chromosome[idx2], chromosome[idx1]
    return chromosome
# PROCESS 6: TERMINATION CONDITION & Genetic Algorithm Main Loop 
def genetic_algorithm():

    population = [create_chromosome() for _ in range(POPULATION_SIZE)]
    best_solution = None
    best_fitness = -1

    for generation in range(MAX_GENERATIONS):

        fitness_scores = [fitness(ind) for ind in population]
        current_best = max(fitness_scores)
        if current_best > best_fitness:
            best_fitness = current_best
            best_solution = population[fitness_scores.index(current_best)]

        # TERMINATION CONDITION: Check if we found a perfect solution
        if best_fitness == MAX_FITNESS:
            print(f"Perfect solution found at generation {generation+1}")
            break

        new_population = []
        while len(new_population) < POPULATION_SIZE:
            parent1, parent2 = tournament_selection(population)
            child = order_crossover(parent1, parent2)
            if random.random() < MUTATION_RATE:
                child = swap_mutation(child)
            new_population.append(child)

        population = new_population

    return best_solution, best_fitness
# SOLUTION: Run GA and Display Result
solution, fit = genetic_algorithm()
print(f"Camera Placement: {solution}")
print(f"Fitness: {fit}")

Perfect solution found at generation 1
Camera Placement: [1, 3, 5, 0, 2, 4, 6]
Fitness: 21


In [17]:
import random

chromosomes = ['C1', 'C2', 'C3', 'C4', 'C5']
fitnesses = [12, 20, 15, 8, 25]
# PROCESS 1: Roulette Wheel Selection
# DESCRIPTION:
# Each individual gets a slice of a roulette wheel proportional to its fitness.
# A random number selects the winner.
def roulette_wheel_selection(chromosomes, fitnesses):
    total = sum(fitnesses)
    pick = random.uniform(0, total)
    cumulative = 0
    for ch, fit in zip(chromosomes, fitnesses):
        cumulative += fit
        if cumulative > pick:
            return ch
    return chromosomes[-1]
# PROCESS 2: Tournament Selection
# DESCRIPTION:
# Randomly pick k individuals, then return the one with the highest fitness.
def tournament_selection(chromosomes, fitnesses, k=3):
    participants = random.sample(list(zip(chromosomes, fitnesses)), k)
    winner = max(participants, key=lambda x: x[1])[0]
    return winner
# PROCESS 3: Rank Selection
# DESCRIPTION:
# Sort by fitness, assign ranks (1 = worst, N = best).
# Probability is proportional to rank (higher rank → higher chance).
def rank_selection(chromosomes, fitnesses):

    paired = sorted(zip(chromosomes, fitnesses), key=lambda x: x[1])
    n = len(paired)
    ranks = list(range(1, n+1))
    total_rank = sum(ranks)
    probs = [r / total_rank for r in ranks]

    pick = random.uniform(0, 1)
    cumulative = 0
    for (ch, _), p in zip(paired, probs):
        cumulative += p
        if cumulative > pick:
            return ch
    return paired[-1][0]
# LAST STEP: Test each selection technique multiple times
print("=== Roulette Wheel Selection (5 runs) ===")
for _ in range(5):
    print(roulette_wheel_selection(chromosomes, fitnesses))

print("\n=== Tournament Selection (5 runs) ===")
for _ in range(5):
    print(tournament_selection(chromosomes, fitnesses))

print("\n=== Rank Selection (5 runs) ===")
for _ in range(5):
    print(rank_selection(chromosomes, fitnesses))

=== Roulette Wheel Selection (5 runs) ===
C5
C5
C5
C3
C4

=== Tournament Selection (5 runs) ===
C5
C5
C2
C5
C5

=== Rank Selection (5 runs) ===
C2
C2
C3
C2
C1


In [18]:
import random

parent1 = [1, 3, 5, 0, 6, 4, 2]
parent2 = [2, 5, 1, 6, 0, 3, 4]
# PROCESS 1: One-Point Crossover (with duplicate repair)
def one_point_crossover(p1, p2):
    n = len(p1)
    point = random.randint(1, n - 1)
    child = p1[:point] + p2[point:]

    seen = set()
    duplicates = []
    for i, val in enumerate(child):
        if val in seen:
            duplicates.append(i)
        else:
            seen.add(val)
    missing = [x for x in range(n) if x not in seen]
    for idx, new_val in zip(duplicates, missing):
        child[idx] = new_val
    return child
# PROCESS 2: Two-Point Crossover (with duplicate repair)
def two_point_crossover(p1, p2):
    n = len(p1)
    points = sorted(random.sample(range(1, n), 2))
    child = p1[:points[0]] + p2[points[0]:points[1]] + p1[points[1]:]

    seen = set()
    duplicates = []
    for i, val in enumerate(child):
        if val in seen:
            duplicates.append(i)
        else:
            seen.add(val)
    missing = [x for x in range(n) if x not in seen]
    for idx, new_val in zip(duplicates, missing):
        child[idx] = new_val
    return child
# PROCESS 3: Order Crossover (OX)
def order_crossover(p1, p2):
    n = len(p1)
    start, end = sorted(random.sample(range(n), 2))
    child = [None] * n

    child[start:end+1] = p1[start:end+1]

    current = 0
    for gene in p2:
        if gene not in child:
            while child[current] is not None:
                current += 1
            child[current] = gene
    return child
# LAST STEP: Test each crossover technique
print("One-Point Crossover:", one_point_crossover(parent1, parent2))
print("Two-Point Crossover:", two_point_crossover(parent1, parent2))
print("Order Crossover:", order_crossover(parent1, parent2))

One-Point Crossover: [1, 5, 2, 6, 0, 3, 4]
Two-Point Crossover: [1, 3, 5, 6, 0, 4, 2]
Order Crossover: [1, 3, 5, 2, 6, 0, 4]


In [19]:
import random

chromosome = [1, 3, 5, 0, 6, 4, 2]
# PROCESS 1: Swap Mutation
def swap_mutation(chromo):
    idx1, idx2 = random.sample(range(len(chromo)), 2)
    chromo[idx1], chromo[idx2] = chromo[idx2], chromo[idx1]
    return chromo
# PROCESS 2: Scramble Mutation
def scramble_mutation(chromo):
    start = random.randint(0, len(chromo) - 1)
    end = random.randint(start, len(chromo) - 1)
    segment = chromo[start:end+1]
    random.shuffle(segment)
    chromo[start:end+1] = segment
    return chromo
# PROCESS 3: Inversion Mutation
def inversion_mutation(chromo):
    start = random.randint(0, len(chromo) - 1)
    end = random.randint(start, len(chromo) - 1)
    chromo[start:end+1] = reversed(chromo[start:end+1])
    return chromo
# LAST STEP: Test each mutation technique
print("Swap Mutation:", swap_mutation(chromosome.copy()))
print("Scramble Mutation:", scramble_mutation(chromosome.copy()))
print("Inversion Mutation:", inversion_mutation(chromosome.copy()))

Swap Mutation: [1, 3, 5, 0, 2, 4, 6]
Scramble Mutation: [1, 3, 5, 0, 6, 4, 2]
Inversion Mutation: [1, 3, 5, 4, 6, 0, 2]


In [1]:
import random
#import sys

# ======================== Problem Definition ========================
N = 8                         # board size (N-Queens)
POP_SIZE = 100
MAX_GEN = 500
MUTATION_RATE = 0.1
TOURNAMENT_SIZE = 3

PERFECT_FITNESS = N * (N - 1) // 2

# ======================== Chromosome ========================
def create_chromosome():
    return random.sample(range(N), N)

# ======================== Fitness ========================
def fitness(chromosome):
    attacks = 0
    for i in range(N):
        for j in range(i + 1, N):
            if abs(chromosome[i] - chromosome[j]) == abs(i - j):
                attacks += 1
    return PERFECT_FITNESS - attacks

# ======================== Selection ========================
def roulette_wheel_selection(population, fitnesses):
    total = sum(fitnesses)
    if total == 0:
        selected = random.sample(population, 2)
        return selected[0], selected[1]
    # first parent
    pick = random.uniform(0, total)
    cumulative = 0
    for i, fit in enumerate(fitnesses):
        cumulative += fit
        if cumulative > pick:
            parent1 = population[i]
            break
    # second parent
    pick = random.uniform(0, total)
    cumulative = 0
    for i, fit in enumerate(fitnesses):
        cumulative += fit
        if cumulative > pick:
            parent2 = population[i]
            break
    return parent1, parent2

def tournament_selection(population, fitnesses, k=TOURNAMENT_SIZE):
    parents = []
    for _ in range(2):
        participants = random.sample(list(zip(population, fitnesses)), k)
        winner = max(participants, key=lambda x: x[1])[0]
        parents.append(winner)
    return parents[0], parents[1]

def rank_selection(population, fitnesses):
    paired = sorted(zip(population, fitnesses), key=lambda x: x[1])
    n = len(paired)
    ranks = list(range(1, n + 1))
    total_rank = sum(ranks)
    probs = [r / total_rank for r in ranks]

    def select_one():
        pick = random.uniform(0, 1)
        cumulative = 0
        for i, (ind, _) in enumerate(paired):
            cumulative += probs[i]
            if cumulative > pick:
                return ind
        return paired[-1][0]

    return select_one(), select_one()

# ======================== Crossover ========================
def repair_duplicates(child):
    n = len(child)
    seen = set()
    duplicates = []
    for i, val in enumerate(child):
        if val in seen:
            duplicates.append(i)
        else:
            seen.add(val)
    missing = [x for x in range(n) if x not in seen]
    for idx, new_val in zip(duplicates, missing):
        child[idx] = new_val
    return child

def one_point_crossover(p1, p2):
    point = random.randint(1, N - 1)
    child = p1[:point] + p2[point:]
    return repair_duplicates(child)

def two_point_crossover(p1, p2):
    points = sorted(random.sample(range(1, N), 2))
    child = p1[:points[0]] + p2[points[0]:points[1]] + p1[points[1]:]
    return repair_duplicates(child)

def order_crossover(p1, p2):
    start, end = sorted(random.sample(range(N), 2))
    child = [None] * N
    child[start:end + 1] = p1[start:end + 1]
    current = 0
    for gene in p2:
        if gene not in child:
            while child[current] is not None:
                current += 1
            child[current] = gene
    return child

# ======================== Mutation ========================
def swap_mutation(chromo):
    i, j = random.sample(range(N), 2)
    chromo[i], chromo[j] = chromo[j], chromo[i]
    return chromo

def scramble_mutation(chromo):
    start = random.randint(0, N - 1)
    end = random.randint(start, N - 1)
    segment = chromo[start:end + 1]
    random.shuffle(segment)
    chromo[start:end + 1] = segment
    return chromo

def inversion_mutation(chromo):
    start = random.randint(0, N - 1)
    end = random.randint(start, N - 1)
    chromo[start:end + 1] = reversed(chromo[start:end + 1])
    return chromo

# ======================== GA Runner ========================
def genetic_algorithm(selection_func, crossover_func, mutation_func):
    random.seed(42)          # fixed seed for consistency
    population = [create_chromosome() for _ in range(POP_SIZE)]
    best_fitness = -1
    best_solution = None
    generation_found = MAX_GEN

    for generation in range(MAX_GEN):
        fitnesses = [fitness(ind) for ind in population]
        current_best = max(fitnesses)
        if current_best > best_fitness:
            best_fitness = current_best
            best_solution = population[fitnesses.index(current_best)]
            if best_fitness == PERFECT_FITNESS:
                generation_found = generation + 1
                break

        new_pop = []
        while len(new_pop) < POP_SIZE:
            p1, p2 = selection_func(population, fitnesses)
            child = crossover_func(p1, p2)
            if random.random() < MUTATION_RATE:
                child = mutation_func(child)
            new_pop.append(child)
        population = new_pop

    return best_solution, best_fitness, generation_found

# ======================== Main: Explicit Calls (no loops) ========================
def main():
    print("=" * 80)
    print("Running Genetic Algorithm for N-Queens (N={}) with all combinations".format(N))
    print("=" * 80)
    print(f"{'Selection':<10} | {'Crossover':<10} | {'Mutation':<10} | {'Gen found':<10} | {'Fitness':<10}")
    print("-" * 70)

    # We'll collect results manually
    results = []

    # ---- Roulette with One-point ----
    solution, fit, gen = genetic_algorithm(roulette_wheel_selection, one_point_crossover, swap_mutation)
    results.append(("Roulette", "One-point", "Swap", gen, fit))
    print("Roulette   | One-point  | Swap       | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(roulette_wheel_selection, one_point_crossover, scramble_mutation)
    results.append(("Roulette", "One-point", "Scramble", gen, fit))
    print("Roulette   | One-point  | Scramble   | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(roulette_wheel_selection, one_point_crossover, inversion_mutation)
    results.append(("Roulette", "One-point", "Inversion", gen, fit))
    print("Roulette   | One-point  | Inversion  | {:<10} | {:<10}".format(gen, fit))

    # ---- Roulette with Two-point ----
    solution, fit, gen = genetic_algorithm(roulette_wheel_selection, two_point_crossover, swap_mutation)
    results.append(("Roulette", "Two-point", "Swap", gen, fit))
    print("Roulette   | Two-point  | Swap       | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(roulette_wheel_selection, two_point_crossover, scramble_mutation)
    results.append(("Roulette", "Two-point", "Scramble", gen, fit))
    print("Roulette   | Two-point  | Scramble   | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(roulette_wheel_selection, two_point_crossover, inversion_mutation)
    results.append(("Roulette", "Two-point", "Inversion", gen, fit))
    print("Roulette   | Two-point  | Inversion  | {:<10} | {:<10}".format(gen, fit))

    # ---- Roulette with Order ----
    solution, fit, gen = genetic_algorithm(roulette_wheel_selection, order_crossover, swap_mutation)
    results.append(("Roulette", "Order", "Swap", gen, fit))
    print("Roulette   | Order      | Swap       | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(roulette_wheel_selection, order_crossover, scramble_mutation)
    results.append(("Roulette", "Order", "Scramble", gen, fit))
    print("Roulette   | Order      | Scramble   | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(roulette_wheel_selection, order_crossover, inversion_mutation)
    results.append(("Roulette", "Order", "Inversion", gen, fit))
    print("Roulette   | Order      | Inversion  | {:<10} | {:<10}".format(gen, fit))

    # ---- Tournament with One-point ----
    solution, fit, gen = genetic_algorithm(tournament_selection, one_point_crossover, swap_mutation)
    results.append(("Tournament", "One-point", "Swap", gen, fit))
    print("Tournament | One-point  | Swap       | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(tournament_selection, one_point_crossover, scramble_mutation)
    results.append(("Tournament", "One-point", "Scramble", gen, fit))
    print("Tournament | One-point  | Scramble   | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(tournament_selection, one_point_crossover, inversion_mutation)
    results.append(("Tournament", "One-point", "Inversion", gen, fit))
    print("Tournament | One-point  | Inversion  | {:<10} | {:<10}".format(gen, fit))

    # ---- Tournament with Two-point ----
    solution, fit, gen = genetic_algorithm(tournament_selection, two_point_crossover, swap_mutation)
    results.append(("Tournament", "Two-point", "Swap", gen, fit))
    print("Tournament | Two-point  | Swap       | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(tournament_selection, two_point_crossover, scramble_mutation)
    results.append(("Tournament", "Two-point", "Scramble", gen, fit))
    print("Tournament | Two-point  | Scramble   | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(tournament_selection, two_point_crossover, inversion_mutation)
    results.append(("Tournament", "Two-point", "Inversion", gen, fit))
    print("Tournament | Two-point  | Inversion  | {:<10} | {:<10}".format(gen, fit))

    # ---- Tournament with Order ----
    solution, fit, gen = genetic_algorithm(tournament_selection, order_crossover, swap_mutation)
    results.append(("Tournament", "Order", "Swap", gen, fit))
    print("Tournament | Order      | Swap       | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(tournament_selection, order_crossover, scramble_mutation)
    results.append(("Tournament", "Order", "Scramble", gen, fit))
    print("Tournament | Order      | Scramble   | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(tournament_selection, order_crossover, inversion_mutation)
    results.append(("Tournament", "Order", "Inversion", gen, fit))
    print("Tournament | Order      | Inversion  | {:<10} | {:<10}".format(gen, fit))

    # ---- Rank with One-point ----
    solution, fit, gen = genetic_algorithm(rank_selection, one_point_crossover, swap_mutation)
    results.append(("Rank", "One-point", "Swap", gen, fit))
    print("Rank       | One-point  | Swap       | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(rank_selection, one_point_crossover, scramble_mutation)
    results.append(("Rank", "One-point", "Scramble", gen, fit))
    print("Rank       | One-point  | Scramble   | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(rank_selection, one_point_crossover, inversion_mutation)
    results.append(("Rank", "One-point", "Inversion", gen, fit))
    print("Rank       | One-point  | Inversion  | {:<10} | {:<10}".format(gen, fit))

    # ---- Rank with Two-point ----
    solution, fit, gen = genetic_algorithm(rank_selection, two_point_crossover, swap_mutation)
    results.append(("Rank", "Two-point", "Swap", gen, fit))
    print("Rank       | Two-point  | Swap       | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(rank_selection, two_point_crossover, scramble_mutation)
    results.append(("Rank", "Two-point", "Scramble", gen, fit))
    print("Rank       | Two-point  | Scramble   | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(rank_selection, two_point_crossover, inversion_mutation)
    results.append(("Rank", "Two-point", "Inversion", gen, fit))
    print("Rank       | Two-point  | Inversion  | {:<10} | {:<10}".format(gen, fit))

    # ---- Rank with Order ----
    solution, fit, gen = genetic_algorithm(rank_selection, order_crossover, swap_mutation)
    results.append(("Rank", "Order", "Swap", gen, fit))
    print("Rank       | Order      | Swap       | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(rank_selection, order_crossover, scramble_mutation)
    results.append(("Rank", "Order", "Scramble", gen, fit))
    print("Rank       | Order      | Scramble   | {:<10} | {:<10}".format(gen, fit))

    solution, fit, gen = genetic_algorithm(rank_selection, order_crossover, inversion_mutation)
    results.append(("Rank", "Order", "Inversion", gen, fit))
    print("Rank       | Order      | Inversion  | {:<10} | {:<10}".format(gen, fit))

    print("-" * 70)
    print("\nSummary: Perfect solution found in {} out of {} combinations.".format(
        sum(1 for r in results if r[4] == PERFECT_FITNESS), len(results)))
    print("Best fitness across all: {}".format(max(r[4] for r in results)))

if __name__ == "__main__":
    main()

Running Genetic Algorithm for N-Queens (N=8) with all combinations
Selection  | Crossover  | Mutation   | Gen found  | Fitness   
----------------------------------------------------------------------
Roulette   | One-point  | Swap       | 5          | 28        
Roulette   | One-point  | Scramble   | 26         | 28        
Roulette   | One-point  | Inversion  | 15         | 28        
Roulette   | Two-point  | Swap       | 3          | 28        
Roulette   | Two-point  | Scramble   | 5          | 28        
Roulette   | Two-point  | Inversion  | 5          | 28        
Roulette   | Order      | Swap       | 2          | 28        
Roulette   | Order      | Scramble   | 3          | 28        
Roulette   | Order      | Inversion  | 2          | 28        
Tournament | One-point  | Swap       | 4          | 28        
Tournament | One-point  | Scramble   | 2          | 28        
Tournament | One-point  | Inversion  | 3          | 28        
Tournament | Two-point  | Swap       | 7   